In [ ]:
import pandas as pd
import lyricsgenius
import time
import os

def add_genius_lyrics():
    GENIUS_TOKEN = "kWb4I3RWTsEEhLAjZudMvcRvVorFA-OUoe-J7UDPY8pVRyaB2dX1OL91vj9kFIFw"
    
    # Initialize Genius API
    genius = lyricsgenius.Genius(GENIUS_TOKEN, timeout=15, retries=3)
    genius.verbose = False 
    # Removes tags like [Chorus] or [Verse 1] from the text
    genius.remove_section_headers = True 

    input_file = 'songs_original.csv'
    output_file = 'songs_lyrics.csv'

    # Checking if there already is an output file
    if os.path.exists(output_file):
        print(f"Found existing file: {output_file}. Continuing search...")
        df = pd.read_csv(output_file)
    else:
        df = pd.read_csv(input_file)
        if 'Lyrics' not in df.columns:
            df['Lyrics'] = None

    print(f"Total of {len(df)} songs. Starting download...\n")

    for index, row in df.iterrows():
        # Skipping if lyrics already present
        if pd.notna(row['Lyrics']) and str(row['Lyrics']).strip() != "":
            continue

        artist = str(row['Artist'])
        title = str(row['Title'])
        
        print(f"[{index + 1}/{len(df)}] Searching: {artist} - {title} ... ", end="")
        
        try:
            song = genius.search_song(title, artist)
            if song:
                df.at[index, 'Lyrics'] = song.lyrics.replace('Embed', '').strip()
                print("Found!")
            else:
                df.at[index, 'Lyrics'] = "NOT_FOUND" # Marking cell, so that it doesn't search again
                print("Not found.")
                
        except Exception as e:
            print(f"Error at song {index}: {e}. Saving and pausing...")
            df.to_csv(output_file, index=False, encoding='utf-8')
            time.sleep(30) # Longer pause at errors (for example rate limit)
            continue

        # Autosave every 20 songs
        if (index + 1) % 20 == 0:
            df.to_csv(output_file, index=False, encoding='utf-8')
            print(f"--- Checkpoint: {index + 1} Songs saved ---")

        # Short break for the API
        time.sleep(0.5)

    print("\nSearch finalized. Removing all songs without lyrics...")
    # Remove all NOT_FOUND tags and replace with NA
    df['Lyrics'] = df['Lyrics'].replace("NOT_FOUND", pd.NA)
    # Deleting all rows with NA
    df = df.dropna(subset=['Lyrics'])

    # Final save at the end
    df.to_csv(output_file, index=False, encoding='utf-8')
    print(f"\nDone! All lyrics saved to '{output_file}'.")

if __name__ == "__main__":
    add_genius_lyrics()

Total of 3130 songs. Starting download...

[1/3130] Searching: Roland W. - Monja ... Found!
[2/3130] Searching: John Fred & His Playboy Band - Judy In Disguise (With Glasses) ... Found!
[3/3130] Searching: Bee Gees - Words ... Found!
[4/3130] Searching: The Beatles - Lady Madonna ... Found!
[5/3130] Searching: Tom Jones - Delilah ... Found!
[6/3130] Searching: Engelbert - A Man Without Love ... Found!
[7/3130] Searching: Les Sauterelles - Heavenly Club ... Found!
[8/3130] Searching: The Beatles - Hey Jude ... Found!
[9/3130] Searching: Mary Hopkin - Those Were The Days ... Found!
[10/3130] Searching: Joe Cocker - With A Little Help From My Friends ... Found!
[11/3130] Searching: Barry Ryan - Eloise ... Found!
[12/3130] Searching: The Beatles - Ob-La-Di, Ob-La-Da ... Found!
[13/3130] Searching: Tommy James And The Shondells - Crimson And Clover ... Found!
[14/3130] Searching: Donovan - Atlantis ... Found!
[15/3130] Searching: The Hollies - Sorry Suzanne ... Found!
[16/3130] Searching: T